In [1]:
# pip install keras-tuner

In [2]:
import pandas as pd
import tensorflow
from tensorflow.keras.layers import Dense
from tensorflow.keras.models import Sequential
import keras_tuner as kt
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split

In [3]:
data = pd.read_csv('huge_1M_titanic.csv')

In [4]:
data = data.sample(10000, random_state=42)

In [5]:
data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
987231,988541,0,3,"Name988541, Mr. Surname988541",male,13.0,5,2,315084,44.567495,B57 B59 B63 B66,S
79954,81264,1,1,"Name81264, Miss. Surname81264",female,73.0,1,1,367655,133.025378,NaN,S
567130,568440,0,3,"Name568440, Mr. Surname568440",male,35.0,0,0,C.A. 5547,6.131359,NaN,S
500891,502201,1,1,"Name502201, Mrs. Surname502201",female,45.0,0,0,345764,141.897841,NaN,C
55399,56709,0,3,"Name56709, Mr. Surname56709",male,34.0,0,0,345774,19.546819,NaN,S


In [6]:
data = data.drop(columns = ['PassengerId','Name','Age','Ticket','Cabin'])

In [7]:
data['Embarked'] = data['Embarked'].replace({'S':'Southampton','C':'Chebourg','Q':'Queenstown'})

In [8]:
data.dropna(subset=['Embarked'],inplace = True)

In [9]:
data['Fare'] = data['Fare'].astype('int')

In [10]:
label = LabelEncoder()
onehot = OneHotEncoder(sparse_output = False)

In [11]:
data['Sex'] = label.fit_transform(data['Sex'])

In [12]:
Embarked = onehot.fit_transform(data[['Embarked']])
Embarked = pd.DataFrame(Embarked, columns = onehot.get_feature_names_out())
data = pd.concat([data.drop(columns = ['Embarked']),Embarked], axis=1)

In [13]:
scale = StandardScaler()

In [14]:
num_cols = ['Pclass', 'SibSp', 'Parch', 'Fare']
data[num_cols] = scale.fit_transform(data[num_cols])

In [15]:
data = data.dropna()

In [16]:
X = data.drop(columns = ['Survived'])
y = data['Survived']

In [17]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size = 20, random_state = 42)

In [18]:
X_train,X_valid,y_train,y_valid = train_test_split(X_train,y_train, test_size = 0.2, random_state = 42)

In [19]:
model = Sequential([Dense(64,input_shape = (X_train.shape[1],),activation = 'relu'), #First hidden Layer
            Dense(32,activation = 'relu'), #Second hidden Layer
            Dense(1,activation = 'sigmoid')]) #Output Layer

c:\Users\geeta\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [20]:
opt = tensorflow.keras.optimizers.Adam(learning_rate = 0.01)

In [21]:
model.compile(optimizer = opt , loss = 'binary_crossentropy', metrics = ['accuracy'])

In [22]:
model.fit(X_train,y_train,validation_data = (X_valid,y_valid), epochs = 50)

Epoch 1/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 119ms/step - accuracy: 0.6866 - loss: 0.6399 - val_accuracy: 0.7059 - val_loss: 0.6275
Epoch 2/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.7313 - loss: 0.4996 - val_accuracy: 0.7059 - val_loss: 0.6656
Epoch 3/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.7761 - loss: 0.4442 - val_accuracy: 0.7059 - val_loss: 0.7589
Epoch 4/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 0.7761 - loss: 0.4575 - val_accuracy: 0.7059 - val_loss: 0.7783
Epoch 5/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.8507 - loss: 0.4132 - val_accuracy: 0.6471 - val_loss: 0.6901
Epoch 6/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - accuracy: 0.8507 - loss: 0.3886 - val_accuracy: 0.6471 - val_loss: 0.6220
Epoch 7/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.8507 - loss: 0.3652 - val_accuracy: 0.6471 - val_loss: 0.5670
Epoch 8/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step - accuracy: 0.8657 - loss: 0.3597 - val_accuracy: 0.7059 - val_loss: 0.5244

# HYPER-PARAMETER TUNING

In [ ]:
def build_model(hp):
    model = Sequential([Dense(64, input_shape = (X_train.shape[1],), activation = 'relu'),
                        Dense(32,activation = 'relu'),
                        Dense(1, activation = 'sigmoid')])
    optimizer = hp.Choice('optimizer',values = [])

    model.compile(optimizer = optimizer, loss = 'binary_crossentropy', metrics = ['accuracy'])

    return model

In [24]:
tuner = kt.RandomSearch(build_model, max_trials = 5, objective = 'val_accuracy')

c:\Users\geeta\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [25]:
tuner.search(X_train,y_train, validation_data = (X_test,y_test), epochs = 10)

Trial 5 Complete [00h 00m 02s]
val_accuracy: 0.75

Best val_accuracy So Far: 0.75
Total elapsed time: 00h 00m 14s


In [26]:
tuner.get_best_hyperparameters()[0].values

{'optimizer': 'Adamax'}

In [27]:
model = tuner.get_best_models(num_models = 1)[0]

c:\Users\geeta\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
c:\Users\geeta\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\saving\saving_lib.py:801: UserWarning: Skipping variable loading for optimizer 'adamax', because it has 2 variables whereas the saved optimizer has 14 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [28]:
model.fit(X_train,y_train, validation_data = (X_test,y_test), epochs = 21, initial_epoch = 11)

Epoch 12/21
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 89ms/step - accuracy: 0.7015 - loss: 0.6595 - val_accuracy: 0.6500 - val_loss: 0.6589
Epoch 13/21
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.7910 - loss: 0.6361 - val_accuracy: 0.6500 - val_loss: 0.6496
Epoch 14/21
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.8358 - loss: 0.6201 - val_accuracy: 0.6500 - val_loss: 0.6442
Epoch 15/21
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.8358 - loss: 0.6082 - val_accuracy: 0.6500 - val_loss: 0.6409
Epoch 16/21
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.8358 - loss: 0.5984 - val_accuracy: 0.6500 - val_loss: 0.6380
Epoch 17/21
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.8358 - loss: 0.5906 - val_accuracy: 0.6500 - val_loss: 0.6359
Epoch 18/21
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.8507 - loss: 0.5823 - val_accuracy: 0.6500 - val_loss: 0.6345
Epoch 19/21
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.8358 - loss: 0.5762 - val_accuracy: 0.6000 - val_loss:

## Choosing the most ideal Number of Nodes to have in Hidden Layer

In [29]:
opt = tensorflow.keras.optimizers.Adam(learning_rate = 0.8)
tensorflow.config.run_functions_eagerly(True)
from tensorflow.keras.layers import Input

In [32]:
def build_model(hp):

    nodes = hp.Int('nodes',8,128, step = 8)
    model = Sequential()
    model.add(Input(shape = (X_train.shape[1],)))
    model.add(Dense(nodes, activation = 'relu'))
    model.add(Dense(nodes, activation = 'relu'))
    model.add(Dense(1, activation = 'sigmoid'))

    model.compile(optimizer = 'SGD', metrics = ['accuracy'], loss = 'binary_crossentropy')

    return model


In [38]:
tuner = kt.RandomSearch(build_model, objective = 'val_loss',max_trials = 5, directory = 'nodes', project_name = 'nodes_details')

In [39]:
tuner.search(X_train,y_train, validation_data = (X_test,y_test), epochs = 10)

Trial 5 Complete [00h 00m 03s]
val_loss: 0.6779575943946838

Best val_loss So Far: 0.6282830238342285
Total elapsed time: 00h 00m 14s


In [40]:
tuner.get_best_hyperparameters()[0].values

{'nodes': 16}

In [41]:
model = tuner.get_best_models(num_models = 1)[0]

In [42]:
model.fit(X_train,y_train, validation_data = (X_test,y_test), epochs= 100, initial_epoch = 10)

Epoch 11/100


1/3 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step - accuracy: 0.4062 - loss: 0.7402

c:\Users\geeta\AppData\Local\Programs\Python\Python312\Lib\site-packages\tensorflow\python\data\ops\structured_function.py:258: UserWarning: Even though the `tf.config.experimental_run_functions_eagerly` option is set, this option does not apply to tf.data functions. To force eager execution of tf.data functions, please use `tf.data.experimental.enable_debug_mode()`.
  warnings.warn(
c:\Users\geeta\AppData\Local\Programs\Python\Python312\Lib\site-packages\tensorflow\python\data\ops\structured_function.py:258: UserWarning: Even though the `tf.config.experimental_run_functions_eagerly` option is set, this option does not apply to tf.data functions. To force eager execution of tf.data functions, please use `tf.data.experimental.enable_debug_mode()`.
  warnings.warn(


3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step - accuracy: 0.4776 - loss: 0.7292 - val_accuracy: 0.6500 - val_loss: 0.6287
Epoch 12/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 0.4925 - loss: 0.7176 - val_accuracy: 0.6500 - val_loss: 0.6284
Epoch 13/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 0.5224 - loss: 0.7093 - val_accuracy: 0.6500 - val_loss: 0.6306
Epoch 14/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step - accuracy: 0.5224 - loss: 0.6979 - val_accuracy: 0.6500 - val_loss: 0.6316
Epoch 15/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - accuracy: 0.5224 - loss: 0.6949 - val_accuracy: 0.6500 - val_loss: 0.6321
Epoch 16/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step - accuracy: 0.5373 - loss: 0.6918 - val_accuracy: 0.6500 - val_loss: 0.6322
Epoch 17/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step - accuracy: 0.5373 - loss: 0.6867 - val_accuracy: 0.6500 - val_loss: 0.6328
Epoch 18/100
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - accuracy: 0.5522 - loss: 0.6803 - val_accuracy: 0.6500 - val_loss: 0.63